In [2]:
"""
============================================================
01_extracao_frames.py
Monitoramento Térmico — Polimento de Aço
============================================================
Extrai frames do HDF5 com amostragem estratificada e gera
labels automáticos por temperatura absoluta para classificação
supervisionada (Normal / Atenção / Crítico).

Saídas:
  frames_model_input/         → PNG grayscale 320×240
  frames_visualization/       → PNG colormap inferno
  frames_metadata.json        → estatísticas + label por frame
  dataset_classification/     → estrutura de pastas por classe
      normal/
      atencao/
      critico/
  amostragem_cobertura.png    → gráfico de cobertura

Dependências:
    pip install h5py numpy pillow matplotlib tqdm
============================================================
"""

from pathlib import Path
import h5py
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import shutil

# ─────────────────────────────────────────────────────────
#  CONFIGURAÇÕES — edite aqui
# ─────────────────────────────────────────────────────────
BASE_DIR  = Path(r"C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico")
HDF5_FILE = BASE_DIR / "pi-camera-data-127001-2019-10-14T12-41-20.hdf5"

OUT_MODEL = BASE_DIR / "frames_model_input"
OUT_VIZ   = BASE_DIR / "frames_visualization"
OUT_META  = BASE_DIR / "frames_metadata.json"
OUT_CLS   = BASE_DIR / "dataset_classification"   # estrutura por classe

DATASET_INTERNO = "pi-camera-1"
TAMANHO_SAIDA   = (320, 240)    # (largura, altura)
COLORMAP        = "inferno"

N_TOTAL_ALVO  = 2000   # frames a exportar

FRAC_UNIFORME = 0.40   # 800 frames — cobertura temporal
FRAC_ALTA_STD = 0.25   # 500 frames — variação espacial interna
FRAC_DELTA    = 0.25   # 500 frames — variação entre frames
FRAC_EXTREMOS = 0.10   # 200 frames — temperaturas extremas

# ── Labels por temperatura absoluta (°C) ──────────────────
# Ajuste estes limiares com um especialista do processo real!
# Eles são estimados como percentis da distribuição — veja o log.
# PERCENTIL_ATENCAO: acima disto → "atencao"
# PERCENTIL_CRITICO: acima disto → "critico"
PERCENTIL_ATENCAO = 75   # ~75% dos frames serão "normal"
PERCENTIL_CRITICO = 95   # ~5% serão "critico"
# ─────────────────────────────────────────────────────────

OUT_MODEL.mkdir(parents=True, exist_ok=True)
OUT_VIZ.mkdir(parents=True, exist_ok=True)
for cls in ["normal", "atencao", "critico"]:
    (OUT_CLS / cls).mkdir(parents=True, exist_ok=True)

try:
    cmap = matplotlib.colormaps[COLORMAP]
except AttributeError:
    import matplotlib.cm as _cm
    cmap = _cm.get_cmap(COLORMAP)

print("=" * 60)
print("  EXTRATOR DE FRAMES — MONITORAMENTO TÉRMICO POLIMENTO")
print("=" * 60)
print(f"  Alvo de frames  : {N_TOTAL_ALVO}")
print(f"  Uniforme        : {int(N_TOTAL_ALVO * FRAC_UNIFORME)}")
print(f"  Alta STD        : {int(N_TOTAL_ALVO * FRAC_ALTA_STD)}")
print(f"  Delta           : {int(N_TOTAL_ALVO * FRAC_DELTA)}")
print(f"  Extremos        : {int(N_TOTAL_ALVO * FRAC_EXTREMOS)}")
print()

with h5py.File(HDF5_FILE, "r") as f:
    dados = f[DATASET_INTERNO]
    H, W, TOTAL = dados.shape
    print(f"  Shape HDF5  : ({H}, {W}, {TOTAL})")
    print(f"  Dtype       : {dados.dtype}")
    print()

    # ══════════════════════════════════════════════════════
    #  PASSO 1 — Estatísticas escalares
    # ══════════════════════════════════════════════════════
    print("► Passo 1/4 — Coletando estatísticas de todos os frames...")

    medias  = np.zeros(TOTAL, dtype=np.float32)
    stds    = np.zeros(TOTAL, dtype=np.float32)
    minimos = np.zeros(TOTAL, dtype=np.float32)
    maximos = np.zeros(TOTAL, dtype=np.float32)

    CHUNK = 5000
    for start in tqdm(range(0, TOTAL, CHUNK), unit="chunk", ncols=70):
        end   = min(start + CHUNK, TOTAL)
        bloco = np.array(dados[:, :, start:end], dtype=np.float32)
        medias[start:end]  = np.nanmean(bloco, axis=(0, 1))
        stds[start:end]    = np.nanstd(bloco,  axis=(0, 1))
        minimos[start:end] = np.nanmin(bloco,  axis=(0, 1))
        maximos[start:end] = np.nanmax(bloco,  axis=(0, 1))

    medias_validas  = medias[np.isfinite(medias)]
    minimos_validos = minimos[np.isfinite(minimos)]
    maximos_validos = maximos[np.isfinite(maximos)]

    nan_frames = int(np.sum(~np.isfinite(medias)))
    if nan_frames > 0:
        print(f"\n  ⚠️  {nan_frames} frames com todos pixels NaN — ignorados")

    G_MEAN = float(np.mean(medias_validas))
    G_STD  = float(np.std(medias_validas))
    G_MIN  = float(np.min(minimos_validos))
    G_MAX  = float(np.max(maximos_validos))

    # Limiares automáticos por percentil da distribuição real
    THRESH_ATENCAO = float(np.percentile(medias_validas, PERCENTIL_ATENCAO))
    THRESH_CRITICO = float(np.percentile(medias_validas, PERCENTIL_CRITICO))

    print(f"\n  Média global : {G_MEAN:.4f}°C  ±{G_STD:.4f}")
    print(f"  Range total  : [{G_MIN:.4f}, {G_MAX:.4f}]°C")
    print(f"\n  ─── Limiares de classificação (ajuste se necessário) ───")
    print(f"  Normal   : < {THRESH_ATENCAO:.4f}°C  (p{PERCENTIL_ATENCAO})")
    print(f"  Atenção  : {THRESH_ATENCAO:.4f} – {THRESH_CRITICO:.4f}°C")
    print(f"  Crítico  : > {THRESH_CRITICO:.4f}°C  (p{PERCENTIL_CRITICO})")
    print()
    print("  ⚠️  IMPORTANTE: valide estes limiares com especialista do processo!")

    # ══════════════════════════════════════════════════════
    #  PASSO 2 — Selecionar índices por estrato
    # ══════════════════════════════════════════════════════
    print("\n► Passo 2/4 — Selecionando índices estratificados...")

    # Sanitiza NaNs
    nan_mask = ~np.isfinite(medias)
    if nan_mask.any():
        fill_m = float(np.nanmedian(medias))
        fill_s = float(np.nanmedian(stds))
        medias[nan_mask] = fill_m
        stds[nan_mask]   = fill_s

    n_unif  = int(N_TOTAL_ALVO * FRAC_UNIFORME)
    n_std   = int(N_TOTAL_ALVO * FRAC_ALTA_STD)
    n_delta = int(N_TOTAL_ALVO * FRAC_DELTA)
    n_extr  = N_TOTAL_ALVO - n_unif - n_std - n_delta

    idx_unif = np.linspace(0, TOTAL - 1, n_unif, dtype=int)
    idx_std  = np.argsort(stds)[::-1][:n_std * 3:3]
    delta    = np.abs(np.diff(medias, prepend=medias[0]))
    idx_delt = np.argsort(delta)[::-1][:n_delta * 3:3]
    n_hot    = n_extr // 2
    n_cold   = n_extr - n_hot
    idx_hot  = np.argsort(medias)[::-1][:n_hot  * 3:3]
    idx_cold = np.argsort(medias)[:n_cold * 3:3]
    idx_extr = np.concatenate([idx_hot, idx_cold])

    todos_idx = np.unique(np.concatenate([idx_unif, idx_std, idx_delt, idx_extr]))

    print(f"  Frames selecionados : {len(todos_idx)}")
    print(f"  Cobertura temporal  : {100 * len(todos_idx) / TOTAL:.2f}%")

    idx_std_set  = set(idx_std.tolist())
    idx_delt_set = set(idx_delt.tolist())
    idx_extr_set = set(idx_extr.tolist())

    # ══════════════════════════════════════════════════════
    #  PASSO 3 — Exportar frames + labels
    # ══════════════════════════════════════════════════════
    print(f"\n► Passo 3/4 — Exportando {len(todos_idx)} frames com labels...")

    contagem_labels = {"normal": 0, "atencao": 0, "critico": 0}

    metadados = {
        "dataset"           : DATASET_INTERNO,
        "shape_original"    : [H, W, TOTAL],
        "frames_total"      : int(TOTAL),
        "frames_exportados" : int(len(todos_idx)),
        "normalizacao_visual": "local_por_frame",
        "tamanho_saida"     : list(TAMANHO_SAIDA),
        "colormap"          : COLORMAP,
        "global_stats": {
            "mean_das_medias" : round(G_MEAN, 6),
            "std_das_medias"  : round(G_STD,  6),
            "min_absoluto"    : round(G_MIN,   6),
            "max_absoluto"    : round(G_MAX,   6),
        },
        "limiares_classificacao": {
            "thresh_atencao"   : round(THRESH_ATENCAO, 6),
            "thresh_critico"   : round(THRESH_CRITICO, 6),
            "percentil_atencao": PERCENTIL_ATENCAO,
            "percentil_critico": PERCENTIL_CRITICO,
            "descricao"        : "Ajustar com especialista do processo de polimento",
        },
        "estratos": {
            "uniforme": int(n_unif),
            "alta_std": int(n_std),
            "delta"   : int(n_delta),
            "extremos": int(n_extr),
        },
        "frames": []
    }

    for seq, orig_idx in enumerate(tqdm(todos_idx, unit="frame", ncols=70)):
        frame = np.array(dados[:, :, orig_idx], dtype=np.float32)

        # Trata NaN
        if not np.all(np.isfinite(frame)):
            frame[~np.isfinite(frame)] = np.nanmean(frame) if np.any(np.isfinite(frame)) else 0.0

        f_min  = float(np.min(frame))
        f_max  = float(np.max(frame))
        f_mean = float(np.mean(frame))
        f_std  = float(np.std(frame))

        # ── Label por temperatura absoluta ──
        if f_mean >= THRESH_CRITICO:
            label = "critico"
        elif f_mean >= THRESH_ATENCAO:
            label = "atencao"
        else:
            label = "normal"
        contagem_labels[label] += 1

        # ── Normalização LOCAL (preserva detalhes visuais) ──
        rng = f_max - f_min
        frame_norm = (frame - f_min) / rng if rng > 0 else np.zeros_like(frame)
        frame_norm = np.clip(frame_norm, 0.0, 1.0)

        # Grayscale → model input (canal 1)
        img_gray = Image.fromarray((frame_norm * 255).astype(np.uint8), mode="L")
        img_gray = img_gray.resize(TAMANHO_SAIDA, resample=Image.Resampling.BILINEAR)
        fname = f"frame_{seq:06d}.png"
        img_gray.save(OUT_MODEL / fname)

        # Colormap inferno → visualização
        rgba     = cmap(frame_norm)
        rgb      = (rgba[:, :, :3] * 255).astype(np.uint8)
        img_color = Image.fromarray(rgb, mode="RGB")
        img_color = img_color.resize(TAMANHO_SAIDA, resample=Image.Resampling.BILINEAR)
        img_color.save(OUT_VIZ / fname)

        # Cópia para pasta de classificação (symlink seria melhor em Linux)
        shutil.copy(OUT_MODEL / fname, OUT_CLS / label / fname)

        # Estrato de origem
        if orig_idx in idx_std_set:    estrato = "alta_std"
        elif orig_idx in idx_delt_set: estrato = "delta"
        elif orig_idx in idx_extr_set: estrato = "extremo"
        else:                          estrato = "uniforme"

        mean_norm = (f_mean - G_MIN) / (G_MAX - G_MIN) if (G_MAX - G_MIN) > 0 else 0.0

        metadados["frames"].append({
            "seq_id"          : seq,
            "orig_idx"        : int(orig_idx),
            "estrato"         : estrato,
            "label"           : label,
            "mean_abs"        : round(f_mean, 4),
            "std_abs"         : round(f_std,  4),
            "min_abs"         : round(f_min,  4),
            "max_abs"         : round(f_max,  4),
            "mean_norm_global": round(float(mean_norm), 6),
        })

with open(OUT_META, "w", encoding="utf-8") as jf:
    json.dump(metadados, jf, indent=2, ensure_ascii=False)

# ══════════════════════════════════════════════════════
#  PASSO 4 — Relatório e gráficos
# ══════════════════════════════════════════════════════
print("\n► Passo 4/4 — Gerando gráficos...")

total_exp = len(todos_idx)
print(f"\n  Distribuição de labels:")
for cls, cnt in contagem_labels.items():
    print(f"    {cls:8s}: {cnt:5d} ({100*cnt/total_exp:.1f}%)")

# Aviso de desbalanceamento
if contagem_labels["critico"] < 50:
    print("\n  ⚠️  ATENÇÃO: poucos frames 'critico'. Considere:")
    print("     • Reduzir PERCENTIL_CRITICO (ex: 92)")
    print("     • Aumentar N_TOTAL_ALVO")
    print("     • Usar data augmentation agressivo na classe critico")

medias_arr = medias[todos_idx]
fig, axes = plt.subplots(3, 1, figsize=(14, 9))

# Gráfico 1: temperatura ao longo do tempo com regiões coloridas
axes[0].plot(todos_idx, medias_arr, color="#444", lw=0.5, alpha=0.7)
axes[0].axhspan(G_MIN,          THRESH_ATENCAO, alpha=0.1, color="green",  label="Normal")
axes[0].axhspan(THRESH_ATENCAO, THRESH_CRITICO, alpha=0.1, color="orange", label="Atenção")
axes[0].axhspan(THRESH_CRITICO, G_MAX,          alpha=0.1, color="red",    label="Crítico")
axes[0].axhline(THRESH_ATENCAO, color="orange", lw=1, linestyle="--")
axes[0].axhline(THRESH_CRITICO, color="red",    lw=1, linestyle="--")
axes[0].set_ylabel("Temperatura média (°C)")
axes[0].set_title("Distribuição temporal dos frames selecionados por classe")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)

# Gráfico 2: histograma de temperatura
axes[1].hist(medias_arr, bins=50, color="#0077b6", alpha=0.7, edgecolor="white", lw=0.3)
axes[1].axvline(THRESH_ATENCAO, color="orange", lw=2, linestyle="--", label=f"Limiar atenção ({THRESH_ATENCAO:.2f}°C)")
axes[1].axvline(THRESH_CRITICO, color="red",    lw=2, linestyle="--", label=f"Limiar crítico ({THRESH_CRITICO:.2f}°C)")
axes[1].set_xlabel("Temperatura média (°C)")
axes[1].set_ylabel("Frequência")
axes[1].set_title("Histograma de temperatura — frames selecionados")
axes[1].legend()
axes[1].grid(True, alpha=0.2)

# Gráfico 3: pizza de labels
labels_nomes  = list(contagem_labels.keys())
labels_counts = list(contagem_labels.values())
colors = ["#2a9d5c", "#e9c46a", "#e63946"]
axes[2].pie(labels_counts, labels=[f"{n}\n({c})" for n, c in zip(labels_nomes, labels_counts)],
            colors=colors, autopct="%1.1f%%", startangle=90)
axes[2].set_title("Distribuição de classes (labels)")

plt.tight_layout()
plt.savefig(BASE_DIR / "amostragem_cobertura.png", dpi=130, bbox_inches="tight")
plt.close()

print()
print("=" * 60)
print("  CONCLUÍDO")
print(f"  Frames exportados : {total_exp}")
print(f"  Model input       : {OUT_MODEL}")
print(f"  Visualização      : {OUT_VIZ}")
print(f"  Dataset cls       : {OUT_CLS}")
print(f"  Metadados JSON    : {OUT_META}")
print(f"  Gráfico           : {BASE_DIR / 'amostragem_cobertura.png'}")
print("=" * 60)
print()
print("  PRÓXIMO PASSO: revise os limiares de classificação com")
print("  um especialista do processo antes de treinar o modelo.")
print("  Arquivo: frames_metadata.json → limiares_classificacao")

  EXTRATOR DE FRAMES — MONITORAMENTO TÉRMICO POLIMENTO
  Alvo de frames  : 2000
  Uniforme        : 800
  Alta STD        : 500
  Delta           : 500
  Extremos        : 200

  Shape HDF5  : (24, 32, 332583)
  Dtype       : float32

► Passo 1/4 — Coletando estatísticas de todos os frames...


100%|██████████████████████████████| 67/67 [00:06<00:00, 10.89chunk/s]



  Média global : 17.9316°C  ±1.0654
  Range total  : [-241.4110, 758.8890]°C

  ─── Limiares de classificação (ajuste se necessário) ───
  Normal   : < 18.1373°C  (p75)
  Atenção  : 18.1373 – 20.0532°C
  Crítico  : > 20.0532°C  (p95)

  ⚠️  IMPORTANTE: valide estes limiares com especialista do processo!

► Passo 2/4 — Selecionando índices estratificados...
  Frames selecionados : 1992
  Cobertura temporal  : 0.60%

► Passo 3/4 — Exportando 1992 frames com labels...


100%|██████████████████████████| 1992/1992 [00:26<00:00, 74.18frame/s]



► Passo 4/4 — Gerando gráficos...

  Distribuição de labels:
    normal  :  1022 (51.3%)
    atencao :   264 (13.3%)
    critico :   706 (35.4%)

  CONCLUÍDO
  Frames exportados : 1992
  Model input       : C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico\frames_model_input
  Visualização      : C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico\frames_visualization
  Dataset cls       : C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico\dataset_classification
  Metadados JSON    : C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico\frames_metadata.json
  Gráfico           : C:\Users\cesar.macieira\Desktop\Trabalho\Python_312\monitoramento-termico\amostragem_cobertura.png

  PRÓXIMO PASSO: revise os limiares de classificação com
  um especialista do processo antes de treinar o modelo.
  Arquivo: frames_metadata.json → limiares_classificacao
